# Deploy de Modelos TensorFlow - Versão Local

## Visão Geral

Este notebook demonstra o fluxo completo de criação, treinamento, serialização e deploy de um modelo TensorFlow, adaptado para execução **fora do ambiente OCI**.

### Etapas do Processo:
1. **Preparar** - Criar os artefatos do modelo necessários para o deploy
2. **Verificar** - Simular chamadas ao modelo antes do deploy
3. **Salvar** - Serializar o modelo para disco
4. **Carregar** - Restaurar o modelo salvo
5. **Predizer** - Realizar inferências com o modelo

---

## Conteúdo

- Introdução  
  - Dataset Fashion-MNIST
- Criar um Modelo TensorFlow
- Serialização do Modelo
  - Formato SavedModel
  - Formato H5
  - Formato ONNX (opcional)
- Carregar e Verificar o Modelo
- Realizar Predições
- Criar API REST Local (Flask)
- Referências

## Configuração do Ambiente

Instalação das dependências necessárias (execute apenas se necessário):

In [ ]:
# Descomente e execute se precisar instalar as dependências
# !pip install tensorflow numpy pandas matplotlib flask joblib

In [ ]:
# Importações principais
import os
import json
import logging
import warnings
import tempfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Configuração de logging e warnings
logging.basicConfig(format='%(levelname)s:%(message)s', level=logging.ERROR)
warnings.filterwarnings('ignore')

# Verificar versão do TensorFlow
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Introdução

## Dataset

O **Fashion_MNIST** é um conjunto de dados de imagens de artigos de vestuário da Zalando, consistindo em um conjunto de treinamento com **60.000 exemplos** e um conjunto de teste com **10.000 exemplos**. 

Cada exemplo é uma imagem em escala de cinza de **28x28 pixels**, associada a um rótulo de **10 classes**.

O Fashion-MNIST serve como um substituto direto do dataset original **MNIST**, sendo amplamente utilizado para benchmark de algoritmos de aprendizado de máquina.

### Classes do Dataset:

| Label | Descrição |
|-------|-----------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

Cada imagem possui **28 pixels de altura e 28 pixels de largura**, totalizando **784 pixels**. O valor de cada pixel é um inteiro entre **0 e 255**.

In [ ]:
# Carregando o dataset Fashion-MNIST
fmnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fmnist.load_data()

# Nomes das classes
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Informações sobre o dataset
print(f"Formato dos dados de treino: {x_train.shape}")
print(f"Formato dos dados de teste: {x_test.shape}")
print(f"Número de classes: {len(class_names)}")
print(f"Valores dos pixels: min={x_train.min()}, max={x_train.max()}")

In [ ]:
# Visualizando algumas imagens do conjunto de treino
from matplotlib import pyplot as plt

plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
    
plt.suptitle('Exemplos do Dataset Fashion-MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## Criação de um Modelo TensorFlow

A próxima célula cria uma classe `TFModel` que:
1. Carrega os dados do Fashion-MNIST
2. Normaliza os dados para valores entre 0 e 1
3. Para reduzir o tempo de treinamento, utiliza apenas os primeiros **10.000 exemplos**

### Arquitetura do Modelo:
- **Camada de entrada**: Flatten com 784 nós (28x28)
- **Camada oculta**: Dense com 128 nós e ativação ReLU
- **Dropout**: 20% para reduzir overfitting
- **Camada de saída**: Dense com 10 nós (uma para cada classe)

In [ ]:
class TFModel:
    """Classe para encapsular o modelo TensorFlow para Fashion-MNIST"""
    
    # Carregar e preparar dados
    fmnist = tf.keras.datasets.fashion_mnist
    (x_train, y_train), (x_test, y_test) = fmnist.load_data()
    
    # Normalizar para escala 0-1
    x_train, x_test = x_train / 255.0, x_test / 255.0
    
    # Reduzir tamanho do conjunto de treino para acelerar
    x_train, y_train = x_train[:10000], y_train[:10000]
    
    # Nomes das classes
    class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
    
    def training(self, epochs=5):
        """Cria e treina o modelo"""
        model = tf.keras.models.Sequential([
            tf.keras.layers.Flatten(input_shape=(28, 28)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(10),
        ])
        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        
        model.compile(
            optimizer="adam",
            loss=loss_fn,
            metrics=["accuracy"]
        )
        
        model.fit(
            self.x_train, 
            self.y_train, 
            epochs=epochs,
            validation_split=0.2,
            verbose=1
        )
        
        return model

## Treinamento e Predicao do Modelo

Apos a definicao da classe do modelo, a proxima celula instancia o modelo e executa o treinamento.

Depois que o modelo e treinado, utiliza-se o metodo `.predict()` para realizar predicoes. Cada predicao retorna **10 valores**, um para cada classe possivel. O valor mais alto indica a classe prevista.

In [ ]:
# Instanciar e treinar o modelo
tf_data = TFModel()
model = tf_data.training(epochs=5)

# Exibir resumo do modelo
model.summary()

In [ ]:
# Realizar predicoes em um subconjunto do dataset de teste
predictions = model.predict(tf_data.x_test[:5])

print("Predicoes brutas (logits):")
print(predictions)

print("\nClasses preditas:")
predicted_classes = np.argmax(predictions, axis=1)
for i, pred in enumerate(predicted_classes):
    print(f"  Exemplo {i}: {tf_data.class_names[pred]} (real: {tf_data.class_names[tf_data.y_test[i]]})")

In [ ]:
# Avaliar o modelo no conjunto de teste completo
test_loss, test_accuracy = model.evaluate(tf_data.x_test, tf_data.y_test, verbose=0)
print(f"Acuracia no conjunto de teste: {test_accuracy:.4f}")
print(f"Loss no conjunto de teste: {test_loss:.4f}")

## Serializacao do Modelo TensorFlow

O TensorFlow oferece diversos formatos para salvar modelos. A seguir, vamos explorar os principais:

### Formatos disponiveis:

1. **SavedModel** (recomendado): Formato nativo do TensorFlow, inclui grafo e pesos
2. **H5/HDF5**: Formato Keras tradicional, arquivo unico
3. **ONNX**: Formato interoperavel (opcional, requer tf2onnx)

### Arquivos gerados na preparacao:

- **model.h5** ou **saved_model/**: Modelo serializado
- **metadata.json**: Metadados do modelo (schema, versao, etc.)
- **score.py**: Script para inferencia

In [ ]:
# Criar diretorio para artefatos do modelo
artifact_dir = Path("./model_artifacts")
artifact_dir.mkdir(exist_ok=True)

print(f"Diretorio de artefatos: {artifact_dir.absolute()}")

### Classe LocalTensorFlowModel

A classe abaixo replica a funcionalidade do `TensorFlowModel` do ADS para uso local, incluindo os metodos:
- `prepare()` - Prepara os artefatos do modelo
- `verify()` - Verifica se o modelo funciona corretamente
- `save()` - Salva o modelo em disco
- `summary_status()` - Exibe o status do deploy

In [ ]:
class LocalTensorFlowModel:
    """
    Classe para gerenciar modelos TensorFlow localmente,
    replicando a funcionalidade do TensorFlowModel do OCI ADS.
    """
    
    def __init__(self, estimator, artifact_dir):
        self.estimator = estimator
        self.artifact_dir = Path(artifact_dir)
        self.artifact_dir.mkdir(exist_ok=True)
        
        # Status tracking
        self._status = {
            'initiate': {'status': 'Done', 'details': 'Model initialized'},
            'prepare': {'status': 'Pending', 'details': ''},
            'verify': {'status': 'Pending', 'details': ''},
            'save': {'status': 'Pending', 'details': ''},
            'deploy': {'status': 'Pending', 'details': ''},
        }
        
        # Metadata storage
        self.metadata = {}
        self.input_schema = None
        self.output_schema = None
        
    def summary_status(self):
        """Retorna um DataFrame com o status do processo de deploy"""
        data = []
        for step, info in self._status.items():
            data.append({
                'Step': step.capitalize(),
                'Status': info['status'],
                'Details': info['details']
            })
        return pd.DataFrame(data)
    
    def prepare(self, X_sample=None, y_sample=None, use_case_type=None, 
                model_file_name='model.h5', save_format='h5'):
        """
        Prepara os artefatos do modelo para deploy.
        
        Arquivos gerados:
        - model.h5 ou saved_model/: Modelo serializado
        - input_schema.json: Schema dos dados de entrada
        - output_schema.json: Schema dos dados de saida
        - runtime.yaml: Configuracoes de runtime
        - score.py: Script de inferencia
        - metadata.json: Metadados do modelo
        """
        try:
            # Salvar modelo no formato especificado
            model_path = self.artifact_dir / model_file_name
            
            if save_format == 'h5':
                self.estimator.save(str(model_path))
            else:  # SavedModel format
                saved_model_path = self.artifact_dir / 'saved_model'
                self.estimator.save(str(saved_model_path))
            
            # Criar input schema
            if X_sample is not None:
                self.input_schema = {
                    'dtype': str(X_sample.dtype),
                    'shape': list(X_sample.shape[1:]),
                    'min_value': float(X_sample.min()),
                    'max_value': float(X_sample.max()),
                }
                with open(self.artifact_dir / 'input_schema.json', 'w') as f:
                    json.dump(self.input_schema, f, indent=2)
            
            # Criar output schema
            if y_sample is not None:
                unique_classes = np.unique(y_sample)
                self.output_schema = {
                    'dtype': str(y_sample.dtype),
                    'num_classes': len(unique_classes),
                    'classes': unique_classes.tolist(),
                }
                with open(self.artifact_dir / 'output_schema.json', 'w') as f:
                    json.dump(self.output_schema, f, indent=2)
            
            # Criar runtime.yaml
            runtime_info = {
                'MODEL_ARTIFACT_VERSION': '1.0',
                'MODEL_DEPLOYMENT': {
                    'INFERENCE_PYTHON_VERSION': f'{tf.version.VERSION}',
                },
                'MODEL_PROVENANCE': {
                    'TRAINING_CONDA_ENV': 'local_python',
                    'INFERENCE_CONDA_ENV': 'local_python',
                }
            }
            
            import yaml
            with open(self.artifact_dir / 'runtime.yaml', 'w') as f:
                yaml.dump(runtime_info, f, default_flow_style=False)
            
            # Criar score.py
            score_py_content = '''"""
Score script para inferencia do modelo TensorFlow.
"""
import json
import numpy as np
import tensorflow as tf
from pathlib import Path

model = None
model_path = None

def load_model(model_dir=None):
    """Carrega o modelo do disco."""
    global model, model_path
    
    if model_dir is None:
        model_dir = Path(__file__).parent
    else:
        model_dir = Path(model_dir)
    
    # Tentar carregar H5 primeiro, depois SavedModel
    h5_path = model_dir / "model.h5"
    saved_model_path = model_dir / "saved_model"
    
    if h5_path.exists():
        model = tf.keras.models.load_model(str(h5_path))
        model_path = h5_path
    elif saved_model_path.exists():
        model = tf.keras.models.load_model(str(saved_model_path))
        model_path = saved_model_path
    else:
        raise FileNotFoundError(f"Modelo nao encontrado em {model_dir}")
    
    return model

def predict(data, model=None):
    """
    Realiza predicao com o modelo.
    
    Args:
        data: numpy array ou lista com os dados de entrada
        model: modelo carregado (opcional, usa global se nao fornecido)
    
    Returns:
        Predicoes do modelo
    """
    if model is None:
        model = load_model()
    
    # Converter para numpy array se necessario
    if isinstance(data, list):
        data = np.array(data)
    
    # Realizar predicao
    predictions = model.predict(data)
    
    return predictions.tolist()

if __name__ == "__main__":
    # Teste local
    model = load_model()
    print(f"Modelo carregado de: {model_path}")
    print(model.summary())
'''
            with open(self.artifact_dir / 'score.py', 'w') as f:
                f.write(score_py_content)
            
            # Criar metadata.json
            self.metadata = {
                'model_name': 'TensorFlowModel',
                'framework': 'TensorFlow',
                'framework_version': tf.__version__,
                'use_case_type': use_case_type or 'classification',
                'created_at': datetime.now().isoformat(),
                'input_schema': self.input_schema,
                'output_schema': self.output_schema,
            }
            with open(self.artifact_dir / 'metadata.json', 'w') as f:
                json.dump(self.metadata, f, indent=2)
            
            self._status['prepare'] = {
                'status': 'Done',
                'details': f'Artifacts saved to {self.artifact_dir}'
            }
            
            print(f"Artefatos preparados em: {self.artifact_dir}")
            print(f"Arquivos criados: {list(self.artifact_dir.iterdir())}")
            
        except Exception as e:
            self._status['prepare'] = {
                'status': 'Error',
                'details': str(e)
            }
            raise
        
        return self
    
    def verify(self, X_test):
        """
        Verifica se o modelo funciona corretamente simulando uma predicao.
        
        Args:
            X_test: Dados de teste para verificacao
        
        Returns:
            Predicoes do modelo
        """
        try:
            # Importar e usar o score.py gerado
            import importlib.util
            spec = importlib.util.spec_from_file_location(
                "score", 
                self.artifact_dir / 'score.py'
            )
            score_module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(score_module)
            
            # Carregar modelo e fazer predicao
            loaded_model = score_module.load_model(str(self.artifact_dir))
            predictions = score_module.predict(X_test, loaded_model)
            
            self._status['verify'] = {
                'status': 'Done',
                'details': f'Verification passed with {len(predictions)} predictions'
            }
            
            return predictions
            
        except Exception as e:
            self._status['verify'] = {
                'status': 'Error',
                'details': str(e)
            }
            raise
    
    def save(self, display_name=None):
        """
        Salva o modelo (localmente, ja salvo pelo prepare).
        Retorna um ID simulado.
        
        Args:
            display_name: Nome de exibicao do modelo
        
        Returns:
            model_id: ID unico do modelo
        """
        import hashlib
        import time
        
        # Gerar um ID unico
        model_id = hashlib.md5(
            f"{display_name}_{time.time()}".encode()
        ).hexdigest()[:16]
        
        # Atualizar metadata com display_name
        self.metadata['display_name'] = display_name or 'Unnamed Model'
        self.metadata['model_id'] = model_id
        
        with open(self.artifact_dir / 'metadata.json', 'w') as f:
            json.dump(self.metadata, f, indent=2)
        
        self._status['save'] = {
            'status': 'Done',
            'details': f'Model saved with ID: {model_id}'
        }
        
        print(f"Modelo salvo!")
        print(f"  Display Name: {display_name}")
        print(f"  Model ID: {model_id}")
        print(f"  Location: {self.artifact_dir}")
        
        return model_id
    
    @property
    def runtime_info(self):
        """Retorna informacoes de runtime"""
        runtime_path = self.artifact_dir / 'runtime.yaml'
        if runtime_path.exists():
            import yaml
            with open(runtime_path, 'r') as f:
                return yaml.safe_load(f)
        return None
    
    @property
    def schema_input(self):
        """Retorna o schema de entrada"""
        return self.input_schema
    
    @property
    def schema_output(self):
        """Retorna o schema de saida"""
        return self.output_schema

## Criando o LocalTensorFlowModel

A proxima celula cria o objeto `LocalTensorFlowModel` com o modelo treinado e o diretorio de artefatos.

In [ ]:
# Criar o objeto LocalTensorFlowModel
tf_model = LocalTensorFlowModel(estimator=model, artifact_dir=artifact_dir)

# Verificar status inicial
tf_model.summary_status()

## Prepare

A etapa de preparacao cria os seguintes arquivos:

- **model.h5**: Modelo serializado no formato Keras H5
- **input_schema.json**: Schema dos dados de entrada
- **output_schema.json**: Schema dos dados de saida
- **runtime.yaml**: Configuracoes de runtime
- **score.py**: Script de inferencia
- **metadata.json**: Metadados do modelo

In [ ]:
# Preparar artefatos do modelo
tf_model.prepare(
    X_sample=tf_data.x_test,
    y_sample=tf_data.y_test,
    use_case_type='MULTINOMIAL_CLASSIFICATION',
)

In [ ]:
# Verificar status apos prepare
tf_model.summary_status()

In [ ]:
# Listar artefatos gerados
print("Arquivos criados:")
for f in artifact_dir.iterdir():
    size = f.stat().st_size if f.is_file() else 0
    print(f"  {f.name}: {size:,} bytes")

In [ ]:
# Verificar informacoes de runtime
print("Runtime Info:")
tf_model.runtime_info

In [ ]:
# Verificar schema de entrada
print("Input Schema:")
tf_model.schema_input

## Verify

A etapa **Verify** permite testar o modelo sem necessidade de realizar o deploy completo.

O metodo `.verify()` carrega o modelo usando o `score.py` gerado e realiza predicoes de teste.

In [ ]:
# Verificar o modelo sem realizar deploy
predictions = tf_model.verify(tf_data.x_test[0:3])

print("Predicoes verificadas:")
for i, pred in enumerate(predictions):
    predicted_class = np.argmax(pred)
    print(f"  Exemplo {i}: {tf_data.class_names[predicted_class]}")

In [ ]:
# Verificar status apos verify
tf_model.summary_status()

## Save

Depois de verificar que o modelo funciona corretamente, salvamos os metadados finais.

No OCI, isso enviaria o modelo para o **Model Catalog**. Localmente, apenas atualizamos os metadados e geramos um ID unico.

In [ ]:
# Salvar o modelo
model_id = tf_model.save(display_name="Demo FMNIST TensorFlowModel Local")

In [ ]:
# Verificar status final
tf_model.summary_status()

## Deploy - API REST Local com Flask

Para realizar o deploy localmente, podemos criar uma API REST simples usando Flask.

Esta secao demonstra como criar um endpoint que replica a funcionalidade de um Model Deployment do OCI.

In [ ]:
# Codigo para criar uma API REST local (salvar como app.py para executar)
flask_app_code = '''
"""
API REST para servir o modelo TensorFlow localmente.
Equivalente ao Model Deployment do OCI.

Para executar:
    python app.py

Endpoints:
    POST /predict - Realizar predicao
    GET /health - Verificar saude do servico
"""
from flask import Flask, request, jsonify
import numpy as np
import tensorflow as tf
from pathlib import Path

app = Flask(__name__)

# Carregar modelo
MODEL_DIR = Path("./model_artifacts")
model = None

def load_model():
    global model
    h5_path = MODEL_DIR / "model.h5"
    if h5_path.exists():
        model = tf.keras.models.load_model(str(h5_path))
        print(f"Modelo carregado de {h5_path}")
    return model

@app.route("/health", methods=["GET"])
def health():
    """Endpoint de health check"""
    return jsonify({"status": "healthy", "model_loaded": model is not None})

@app.route("/predict", methods=["POST"])
def predict():
    """
    Endpoint de predicao.
    
    Input esperado (JSON):
        {"data": [[pixel1, pixel2, ..., pixel784], ...]}
        ou
        {"data": [[[28x28 matrix]], ...]}
    
    Output:
        {"predictions": [[class_probs], ...], "classes": [class_idx, ...]}
    """
    try:
        data = request.json.get("data")
        if data is None:
            return jsonify({"error": "Campo 'data' nao encontrado"}), 400
        
        # Converter para numpy array
        input_data = np.array(data)
        
        # Normalizar se necessario (valores 0-255 -> 0-1)
        if input_data.max() > 1:
            input_data = input_data / 255.0
        
        # Fazer predicao
        predictions = model.predict(input_data)
        predicted_classes = np.argmax(predictions, axis=1)
        
        return jsonify({
            "predictions": predictions.tolist(),
            "classes": predicted_classes.tolist()
        })
        
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    load_model()
    print("Iniciando servidor na porta 5000...")
    print("Endpoints disponiveis:")
    print("  GET  http://localhost:5000/health")
    print("  POST http://localhost:5000/predict")
    app.run(host="0.0.0.0", port=5000, debug=True)
'''

# Salvar o codigo da API
api_path = artifact_dir / "app.py"
with open(api_path, 'w') as f:
    f.write(flask_app_code)

print(f"API Flask salva em: {api_path}")
print("\\nPara executar o servidor:")
print(f"  cd {artifact_dir}")
print("  python app.py")

## Predict - Usando o Modelo Salvo

Agora vamos demonstrar como carregar e usar o modelo salvo para fazer predicoes.

In [ ]:
# Carregar o modelo salvo
loaded_model = tf.keras.models.load_model(str(artifact_dir / 'model.h5'))

# Verificar que o modelo foi carregado corretamente
print("Modelo carregado com sucesso!")
loaded_model.summary()

In [ ]:
# Fazer predicoes com o modelo carregado
test_images = tf_data.x_test[:10]
test_labels = tf_data.y_test[:10]

predictions = loaded_model.predict(test_images)
predicted_classes = np.argmax(predictions, axis=1)

# Mostrar resultados
print("Resultados das predicoes:")
print("-" * 50)
for i in range(len(test_images)):
    real = tf_data.class_names[test_labels[i]]
    pred = tf_data.class_names[predicted_classes[i]]
    match = "OK" if test_labels[i] == predicted_classes[i] else "ERRO"
    print(f"Imagem {i}: Real={real:15} | Predito={pred:15} [{match}]")

In [ ]:
# Visualizar as predicoes
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images[i], cmap='gray')
    
    color = 'green' if predicted_classes[i] == test_labels[i] else 'red'
    plt.title(f"Pred: {tf_data.class_names[predicted_classes[i]]}", color=color, fontsize=9)
    plt.xlabel(f"Real: {tf_data.class_names[test_labels[i]]}", fontsize=8)
    plt.xticks([])
    plt.yticks([])

plt.suptitle("Predicoes do Modelo (Verde=Correto, Vermelho=Incorreto)", fontsize=12)
plt.tight_layout()
plt.show()

## Exemplo de Chamada a API (quando o servidor estiver rodando)

Use o codigo abaixo para testar a API quando o servidor Flask estiver em execucao:

In [ ]:
# Exemplo de chamada a API (descomente para usar quando o servidor estiver rodando)
# import requests
# 
# # Preparar dados de teste
# test_data = tf_data.x_test[0:3].tolist()
# 
# # Fazer requisicao POST
# response = requests.post(
#     "http://localhost:5000/predict",
#     json={"data": test_data}
# )
# 
# # Exibir resultado
# result = response.json()
# print("Classes preditas:", result['classes'])
# for i, cls in enumerate(result['classes']):
#     print(f"  Imagem {i}: {tf_data.class_names[cls]}")

## Clean Up

Para limpar os artefatos gerados, execute a celula abaixo:

In [ ]:
# Descomente para limpar os artefatos
# import shutil
# if artifact_dir.exists():
#     shutil.rmtree(artifact_dir)
#     print(f"Diretorio {artifact_dir} removido com sucesso!")

print(f"Artefatos salvos em: {artifact_dir}")
print("Para remover, descomente o codigo acima e execute novamente.")

## Referencias

- [TensorFlow Documentation](https://www.tensorflow.org/guide)
- [Keras Model Saving and Loading](https://www.tensorflow.org/guide/keras/save_and_serialize)
- [Fashion-MNIST Dataset](https://github.com/zalandoresearch/fashion-mnist)
- [Flask Documentation](https://flask.palletsprojects.com/)
- [OCI Data Science - ADS](https://docs.oracle.com/en-us/iaas/tools/ads-sdk/latest/index.html)

---

## Resumo

Este notebook demonstrou o fluxo completo de:

1. **Carregar** o dataset Fashion-MNIST
2. **Criar e treinar** um modelo TensorFlow/Keras
3. **Preparar** os artefatos do modelo (serializar, criar schemas, score.py)
4. **Verificar** que o modelo funciona corretamente
5. **Salvar** o modelo localmente
6. **Deploy** opcional via API REST Flask
7. **Predizer** usando o modelo carregado

Este fluxo replica a funcionalidade do OCI Data Science ADS `TensorFlowModel`, permitindo desenvolvimento e testes locais antes do deploy na nuvem.